# Séance 13 – Système, données et base de données
### Lab : sys, os, json, csv et sqlite3

| | |
|---|---|
| **Durée estimée** | 60 min (2 × 30 min) + bonus |
| **Projet fil rouge** | GestParc – étape 9 : CSV, JSON et persistance SQLite |

**Objectifs**
- Exploiter les arguments, la sortie et les variables d'environnement d'un script
- Parcourir une arborescence de fichiers
- Lire et écrire du JSON et du CSV
- Réaliser un CRUD SQLite avec des requêtes paramétrées, des transactions et des agrégations
- Comprendre et empêcher l'injection SQL

**Mode d'emploi**
- Exécuter une cellule : **Maj + Entrée** ; en cas de comportement étrange : *Redémarrer le noyau et tout exécuter*.
- Complétez les cellules marquées `# TODO`, puis exécutez la cellule **✅ Vérification** qui suit : aucune erreur = réponse correcte.
- Difficulté : ⭐ facile · ⭐⭐ intermédiaire · ⭐⭐⭐ avancé · Indices dans les blocs 💡 repliables.

> ℹ️ Les fichiers sont créés dans `lab_s13/`, à côté du notebook.

In [ ]:
# Cellule de préparation
import csv
import json
import os
import sqlite3
import sys
from pathlib import Path

WORK = Path("lab_s13")
WORK.mkdir(exist_ok=True)
print(sys.version.split()[0], sys.platform, "→", WORK.resolve())

---
## Partie 1 – Système, JSON et CSV

> **Rappel**
> - `sys.argv` (liste de str, `argv[0]` = script), `sys.exit(code)`, `print(..., file=sys.stderr)`
> - `os.environ.get("NOM", "défaut")` · `os.walk(dossier)` → `(dossier, sous_dossiers, fichiers)` · `os.path.join`, `os.path.getsize`
> - `json.dump(obj, f, ensure_ascii=False, indent=2)` / `json.load(f)` · `dumps` / `loads` pour les chaînes
> - `csv.DictReader(f, delimiter=";")` / `csv.DictWriter(f, fieldnames=[...], delimiter=";")` · `open(..., newline="")`

### Exemple guidé : un script qui lit sa configuration dans l'environnement

In [ ]:
os.environ["BACKUP_DIR"] = "/mnt/backup"          # simule un « export BACKUP_DIR=... » du shell

backup_dir = os.environ.get("BACKUP_DIR", "/tmp")
retention = int(os.environ.get("BACKUP_RETENTION_DAYS", "7"))
print(f"Sauvegarde vers {backup_dir}, conservation {retention} jours")
print("Exécuté par :", sys.executable)
print("Erreur simulée (sortie d'erreur)", file=sys.stderr)

### Exercice 1 ⭐ – Analyser les arguments d'une commande

Écrivez `parse_backup_args(argv: list[str]) -> dict` pour la commande
`python backup.py [--dry-run] [--dest DOSSIER] HÔTE [HÔTE ...]` (`argv` a le format de `sys.argv`) :
- `"dry_run"` : booléen ; `"dest"` : dossier (`"/backup"` par défaut) ; `"hosts"` : liste des hôtes ;
- lève `ValueError` si aucun hôte n'est fourni ou si `--dest` n'est suivi d'aucune valeur.

In [ ]:
def parse_backup_args(argv: list[str]) -> dict:
    """Analyse les arguments de la commande backup."""
    # TODO


print(parse_backup_args(["backup.py", "--dry-run", "srv-web-01", "--dest", "/mnt/nas", "srv-db-01"]))

<details><summary>💡 Indice</summary>

Parcourez `argv[1:]` avec un indice (`while`) : quand vous rencontrez `--dest`, la valeur est l'élément suivant. Pour une vraie application : module `argparse` (bonus).

</details>

In [ ]:
# ✅ Vérification
assert parse_backup_args(["backup.py", "--dry-run", "srv-web-01", "--dest", "/mnt/nas", "srv-db-01"]) == \
    {"dry_run": True, "dest": "/mnt/nas", "hosts": ["srv-web-01", "srv-db-01"]}
assert parse_backup_args(["b.py", "srv-01"]) == {"dry_run": False, "dest": "/backup", "hosts": ["srv-01"]}
for bad in [["b.py"], ["b.py", "--dry-run"], ["b.py", "srv-01", "--dest"]]:
    try:
        parse_backup_args(bad)
        raise AssertionError(f"ValueError attendue pour {bad}")
    except ValueError:
        pass
print("✅ Exercice 1 réussi")

### Exercice 2 ⭐⭐ – Configuration par variables d'environnement

Écrivez `load_env_config(env=None) -> dict` (si `env` est `None`, utilisez `os.environ`) qui lit :
- `APP_ENV` (`"dev"` par défaut, doit valoir `dev`, `test` ou `prod`, sinon `ValueError`) ;
- `DB_HOST` (`"localhost"` par défaut) et `DB_PORT` (entier, `5432` par défaut) ;
- `APP_DEBUG` (vrai si la valeur est `1`, `true` ou `yes`, quelle que soit la casse ; faux par défaut) ;
- `DB_PASSWORD` : **obligatoire en `prod`** (sinon `ValueError`), `None` par défaut ailleurs.

Passer un dictionnaire `env` permet de tester la fonction sans modifier l'environnement réel.

In [ ]:
def load_env_config(env: dict[str, str] | None = None) -> dict:
    """Configuration de l'application à partir des variables d'environnement."""
    # TODO


print(load_env_config({"APP_ENV": "test", "DB_PORT": "6543", "APP_DEBUG": "True"}))

<details><summary>💡 Indice</summary>

`os.environ` se comporte comme un dictionnaire : `.get(nom, défaut)`.

</details>

In [ ]:
# ✅ Vérification
assert load_env_config({}) == {"env": "dev", "db_host": "localhost", "db_port": 5432, "debug": False, "db_password": None}
cfg = load_env_config({"APP_ENV": "prod", "DB_HOST": "srv-db-01", "DB_PASSWORD": "s3cr3t", "APP_DEBUG": "YES"})
assert cfg["env"] == "prod" and cfg["db_host"] == "srv-db-01" and cfg["debug"] is True
for bad in [{"APP_ENV": "staging"}, {"APP_ENV": "prod"}, {"DB_PORT": "port"}]:
    try:
        load_env_config(bad)
        raise AssertionError(f"ValueError attendue pour {bad}")
    except ValueError:
        pass
os.environ["APP_ENV"] = "test"
assert load_env_config()["env"] == "test"
print("✅ Exercice 2 réussi")

### Exercice 3 ⭐⭐ – Audit d'une arborescence

La cellule crée une arborescence de projet dans `lab_s13/projet`. Avec `os.walk`, écrivez `audit_tree(root, min_size)` qui renvoie :
- `"files"` : nombre total de fichiers ; `"total_bytes"` : taille totale ;
- `"by_extension"` : extension (en minuscules, `""` si aucune) → nombre de fichiers ;
- `"large_files"` : chemins **relatifs à `root`** (avec `/`) des fichiers de taille ≥ `min_size`, triés.

In [ ]:
def audit_tree(root: str | Path, min_size: int) -> dict:
    """Statistiques d'une arborescence de fichiers."""
    # TODO


root = WORK / "projet"
for relative, size in [("README.md", 120), ("src/app.py", 800), ("src/utils.py", 300), ("data/export.CSV", 5000),
                       ("data/backup/dump.sql", 20000), ("docs/guide.pdf", 15000), ("Makefile", 60)]:
    path = root / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(b"x" * size)
print(audit_tree(root, 10_000))

<details><summary>💡 Indice</summary>

`os.path.splitext("export.CSV")` renvoie `("export", ".CSV")`. `Path(chemin).relative_to(root).as_posix()` donne un chemin relatif avec des `/`.

</details>

In [ ]:
# ✅ Vérification
result = audit_tree(root, 10_000)
assert result["files"] == 7 and result["total_bytes"] == 41_280
assert result["by_extension"] == {".md": 1, ".py": 2, ".csv": 1, ".sql": 1, ".pdf": 1, "": 1}
assert result["large_files"] == ["data/backup/dump.sql", "docs/guide.pdf"]
print("✅ Exercice 3 réussi")

### Exercice 4 ⭐⭐ – Configuration JSON avec dates et ensembles

La configuration `maintenance_plan` contient une `date` et un `set`, que JSON ne sait pas représenter.
1. `save_plan(plan, path)` : convertit la date en texte ISO et l'ensemble en **liste triée**, puis écrit le JSON (UTF-8, `ensure_ascii=False`, `indent=2`) ;
2. `load_plan(path)` : relit le fichier et **reconvertit** la date (`date.fromisoformat`) et l'ensemble.
Le résultat doit être égal à l'objet d'origine.

In [ ]:
from datetime import date

maintenance_plan = {"site": "Sfax", "window_start": date(2026, 11, 7), "hosts": {"srv-web-02", "srv-web-01"},
                    "duration_h": 3, "notify": ["dsi@entreprise.tn"], "rollback": True, "comment": "Mise à jour noyau – redémarrage"}


def save_plan(plan: dict, path: Path) -> None:
    # TODO
    ...


def load_plan(path: Path) -> dict:
    # TODO
    ...


plan_file = WORK / "maintenance.json"
save_plan(maintenance_plan, plan_file)
print(plan_file.read_text(encoding="utf-8"))
print(load_plan(plan_file) == maintenance_plan)

<details><summary>💡 Indice</summary>

`dict(plan)` crée une copie superficielle : on peut remplacer des valeurs sans toucher à l'original.

</details>

In [ ]:
# ✅ Vérification
raw = json.loads(plan_file.read_text(encoding="utf-8"))
assert raw["window_start"] == "2026-11-07" and raw["hosts"] == ["srv-web-01", "srv-web-02"] and raw["rollback"] is True
assert "Mise à jour noyau – redémarrage" in plan_file.read_text(encoding="utf-8"), "ensure_ascii=False attendu"
assert load_plan(plan_file) == maintenance_plan
assert isinstance(maintenance_plan["hosts"], set), "l'objet d'origine ne doit pas être modifié"
print("✅ Exercice 4 réussi")

### Exercice 5 ⭐⭐ – Statistiques depuis un CSV, rapport en CSV

Le fichier `heures.csv` (séparateur `;`) liste les heures passées par collaborateur et par projet.
1. Avec `csv.DictReader`, calculez `hours_by_project` (projet → total d'heures, décimal) et `people_by_project` (projet → ensemble des collaborateurs) ;
2. écrivez avec `csv.DictWriter` le fichier `synthese_projets.csv` : colonnes `projet;heures;collaborateurs;cout`, une ligne par projet (ordre alphabétique),
   coût = heures × 85 DT, collaborateurs = nombre de personnes distinctes.

In [ ]:
hours_file = WORK / "heures.csv"
hours_file.write_text("""collaborateur;projet;date;heures
Amira;Migration AD;2026-10-19;6,5
Karim;Intranet;2026-10-19;4
Amira;Migration AD;2026-10-20;7
Sonia;VPN;2026-10-20;3,5
Karim;Migration AD;2026-10-21;2
Sonia;Intranet;2026-10-21;5
""", encoding="utf-8")

hours_by_project: dict[str, float] = {}
people_by_project: dict[str, set[str]] = {}
# TODO 1

summary_file = WORK / "synthese_projets.csv"
# TODO 2

<details><summary>💡 Indice</summary>

Les heures utilisent une virgule décimale : `float(texte.replace(",", "."))`. `dict.setdefault(clé, set())` renvoie l'ensemble existant ou en crée un.

</details>

In [ ]:
# ✅ Vérification
assert hours_by_project == {"Migration AD": 15.5, "Intranet": 9.0, "VPN": 3.5}
assert people_by_project["Migration AD"] == {"Amira", "Karim"}
with open(summary_file, encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f, delimiter=";"))
assert [r["projet"] for r in rows] == ["Intranet", "Migration AD", "VPN"]
assert rows[1] == {"projet": "Migration AD", "heures": "15.5", "collaborateurs": "2", "cout": "1317.5"}
assert summary_file.read_text(encoding="utf-8").count("\n") == 4
print("✅ Exercice 5 réussi")

---
## Partie 2 – Base de données SQLite

> **Rappel**
> - `conn = sqlite3.connect(fichier ou ":memory:")` · `conn.row_factory = sqlite3.Row` (accès `row["col"]`)
> - `conn.execute(sql, (valeurs,))` avec des `?` · `executemany` · `fetchone()`, `fetchall()` · `cursor.rowcount`, `cursor.lastrowid`
> - `with conn:` = transaction (commit si succès, rollback si exception) · `sqlite3.IntegrityError` pour les contraintes

### Exemple guidé : une table de postes de travail

In [ ]:
conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row
with conn:
    conn.execute("CREATE TABLE workstation (tag TEXT PRIMARY KEY, os TEXT NOT NULL, ram_gb INTEGER)")
    conn.executemany("INSERT INTO workstation VALUES (?, ?, ?)",
                     [("LT-001", "Windows 11", 32), ("LT-002", "Ubuntu 24.04", 16), ("LT-003", "Windows 10", 8)])
to_upgrade = conn.execute("SELECT tag, os FROM workstation WHERE os = ? OR ram_gb < ?", ("Windows 10", 16)).fetchall()
print([dict(row) for row in to_upgrade])
print(conn.execute("SELECT COUNT(*), AVG(ram_gb) FROM workstation").fetchone()[:])
conn.close()

### Exercice 6 ⭐ – Créer une table et l'interroger

Dans une base en mémoire `db` :
1. créez la table `server` : `hostname` (texte, clé primaire), `site` (texte, obligatoire), `cpu` (entier), `ram_gb` (entier) ;
2. insérez les serveurs de `servers` avec `executemany` (dans une transaction `with db:`) ;
3. récupérez dans `tunis_servers` la liste des noms des serveurs du site `"Tunis"` triés par RAM décroissante, et dans `total_ram` la RAM totale (requête `SUM`).

In [ ]:
db = sqlite3.connect(":memory:")
servers = [("srv-web-01", "Tunis", 8, 32), ("srv-db-01", "Tunis", 16, 128), ("srv-web-02", "Sfax", 8, 32), ("srv-app-01", "Tunis", 4, 16)]
# TODO
tunis_servers = ...
total_ram = ...

<details><summary>💡 Indice</summary>

Le paramètre doit être un tuple : `("Tunis",)` avec la virgule.

</details>

In [ ]:
# ✅ Vérification
assert tunis_servers == ["srv-db-01", "srv-web-01", "srv-app-01"]
assert total_ram == 208
try:
    with db:
        db.execute("INSERT INTO server VALUES (?, ?, ?, ?)", ("srv-web-01", "Tunis", 1, 1))
    raise AssertionError("la clé primaire doit empêcher le doublon")
except sqlite3.IntegrityError:
    pass
print("✅ Exercice 6 réussi")

### Exercice 7 ⭐⭐ – Un dépôt CRUD de tickets

Complétez la classe `TicketRepository` (table `ticket` : `id` entier auto-incrémenté, `title`, `priority` parmi P1/P2/P3, `status` = `"NEW"` par défaut, `assignee` facultatif) :
- `create(title, priority) -> int` : renvoie l'identifiant créé (`lastrowid`) ;
- `get(ticket_id) -> dict | None` ;
- `assign(ticket_id, assignee) -> bool` : passe le statut à `"ASSIGNED"` ; `False` si le ticket n'existe pas (`rowcount`) ;
- `close(ticket_id) -> bool` ;
- `list(status=None) -> list[dict]` : tous les tickets (ou ceux d'un statut), triés par priorité puis identifiant ;
- `delete_closed() -> int` : supprime les tickets fermés et renvoie leur nombre.

In [ ]:
class TicketRepository:
    """Accès aux tickets stockés dans SQLite."""

    def __init__(self, db_path: str = ":memory:") -> None:
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row
        with self.conn:
            self.conn.execute("""CREATE TABLE IF NOT EXISTS ticket (
                id INTEGER PRIMARY KEY AUTOINCREMENT,
                title TEXT NOT NULL,
                priority TEXT NOT NULL CHECK (priority IN ('P1', 'P2', 'P3')),
                status TEXT NOT NULL DEFAULT 'NEW',
                assignee TEXT)""")

    # TODO : create, get, assign, close, list, delete_closed


repo = TicketRepository()
for title, priority in [("VPN lent", "P2"), ("Serveur de fichiers arrêté", "P1"), ("Souris HS", "P3")]:
    repo.create(title, priority)
repo.assign(2, "Youssef")
repo.close(3)
print(repo.list())

<details><summary>💡 Indice</summary>

`cursor = conn.execute(...)` puis `cursor.lastrowid` (insertion) ou `cursor.rowcount` (mise à jour, suppression).

</details>

In [ ]:
# ✅ Vérification
r = TicketRepository()
a, b, c = r.create("A", "P2"), r.create("B", "P1"), r.create("C", "P3")
assert (a, b, c) == (1, 2, 3)
assert r.get(1) == {"id": 1, "title": "A", "priority": "P2", "status": "NEW", "assignee": None} and r.get(99) is None
assert r.assign(2, "Youssef") and not r.assign(99, "X") and r.get(2)["status"] == "ASSIGNED"
assert r.close(3) and not r.close(42)
assert [t["id"] for t in r.list()] == [2, 1, 3] and [t["id"] for t in r.list("NEW")] == [1]
assert r.delete_closed() == 1 and r.get(3) is None
try:
    r.create("D", "P9")
    raise AssertionError("la contrainte CHECK doit refuser P9")
except sqlite3.IntegrityError:
    pass
print("✅ Exercice 7 réussi")

### Exercice 8 ⭐⭐ – Attaque puis correction d'une injection SQL

La fonction `login_unsafe` construit sa requête avec une f-string.
1. Trouvez une valeur de `attack_login` qui permet de se connecter en tant qu'`admin` **sans connaître le mot de passe** (n'importe quel mot de passe).
2. Écrivez `login_safe(conn, login, password)` avec une **requête paramétrée** : l'attaque doit échouer.

*(Les mots de passe sont stockés en clair pour simplifier l'exercice : à ne jamais faire en réalité.)*

In [ ]:
users_db = sqlite3.connect(":memory:")
with users_db:
    users_db.execute("CREATE TABLE users (login TEXT PRIMARY KEY, password TEXT, role TEXT)")
    users_db.executemany("INSERT INTO users VALUES (?, ?, ?)",
                         [("admin", "Adm!n-2026", "admin"), ("amira", "Tunis#42", "user")])


def login_unsafe(conn: sqlite3.Connection, login: str, password: str) -> str | None:
    """❌ Version vulnérable : NE PAS REPRODUIRE."""
    sql = f"SELECT role FROM users WHERE login = '{login}' AND password = '{password}'"
    row = conn.execute(sql).fetchone()
    return row[0] if row else None


attack_login = "..."   # TODO 1


def login_safe(conn: sqlite3.Connection, login: str, password: str) -> str | None:
    """✅ Requête paramétrée."""
    # TODO 2


print(login_unsafe(users_db, attack_login, "n'importe quoi"), login_safe(users_db, attack_login, "n'importe quoi"))

<details><summary>💡 Indice</summary>

En SQL, `--` commence un commentaire : tout ce qui suit est ignoré. Fermez la chaîne du login avec une apostrophe.

</details>

In [ ]:
# ✅ Vérification
assert login_unsafe(users_db, attack_login, "mauvais") == "admin", "l'attaque doit réussir sur la version vulnérable"
assert login_safe(users_db, attack_login, "mauvais") is None, "l'attaque doit échouer sur la version sûre"
assert login_safe(users_db, "amira", "Tunis#42") == "user" and login_safe(users_db, "amira", "x") is None
assert login_safe(users_db, "o'brien", "x") is None, "une apostrophe ne doit pas provoquer d'erreur"
print("✅ Exercice 8 réussi")

### Exercice 9 ⭐⭐⭐ – Jointures, agrégations et transaction

La base `helpdesk` contient des employés et des tickets.
1. `stats_by_department` : liste de dictionnaires `{"department", "tickets", "avg_hours"}` (tickets **fermés** uniquement, moyenne arrondie à 1 décimale),
   triée par nombre de tickets décroissant, en **une seule requête** (`JOIN` + `GROUP BY`) ;
2. `top_requester` : nom de l'employé ayant ouvert le plus de tickets (tous statuts) ;
3. `reassign_all(conn, from_tech, to_tech)` : dans **une transaction**, réaffecte tous les tickets ouverts d'un technicien à un autre
   **et** enregistre l'opération dans la table `audit` ; si `to_tech` est vide, lever `ValueError` **à l'intérieur** de la transaction (rien ne doit être modifié).

In [ ]:
helpdesk = sqlite3.connect(":memory:")
helpdesk.row_factory = sqlite3.Row
with helpdesk:
    helpdesk.executescript("""
        CREATE TABLE employee (id INTEGER PRIMARY KEY, name TEXT, department TEXT);
        CREATE TABLE ticket (id INTEGER PRIMARY KEY, requester_id INTEGER REFERENCES employee(id),
                             tech TEXT, status TEXT, hours REAL);
        CREATE TABLE audit (id INTEGER PRIMARY KEY AUTOINCREMENT, action TEXT);
        INSERT INTO employee VALUES (1, 'Amira', 'DSI'), (2, 'Karim', 'Finance'), (3, 'Sonia', 'DSI'), (4, 'Leila', 'RH');
        INSERT INTO ticket VALUES
            (1, 1, 'Youssef', 'CLOSED', 2.5), (2, 2, 'Youssef', 'CLOSED', 1.0), (3, 3, 'Hedi', 'CLOSED', 4.0),
            (4, 1, 'Hedi', 'OPEN', NULL), (5, 2, 'Youssef', 'OPEN', NULL), (6, 1, 'Youssef', 'CLOSED', 3.0),
            (7, 4, 'Hedi', 'CLOSED', 0.5), (8, 2, 'Youssef', 'CLOSED', 2.0), (9, 1, 'Youssef', 'OPEN', NULL);
    """)
# TODO 1 et 2
stats_by_department = ...
top_requester = ...


def reassign_all(conn: sqlite3.Connection, from_tech: str, to_tech: str) -> int:
    """Réaffecte les tickets ouverts d'un technicien ; tout ou rien."""
    # TODO 3


print(stats_by_department, top_requester)
print(reassign_all(helpdesk, "Youssef", "Hedi"))

<details><summary>💡 Indice</summary>

Dans `with conn:`, une exception levée après les `execute` provoque un `rollback` de **toutes** les modifications du bloc.

</details>

In [ ]:
# ✅ Vérification
assert stats_by_department == [{"department": "DSI", "tickets": 3, "avg_hours": 3.2},
                               {"department": "Finance", "tickets": 2, "avg_hours": 1.5},
                               {"department": "RH", "tickets": 1, "avg_hours": 0.5}]
assert top_requester == "Amira"
before = [tuple(r) for r in helpdesk.execute("SELECT id, tech FROM ticket ORDER BY id")]
try:
    reassign_all(helpdesk, "Hedi", "  ")
    raise AssertionError("ValueError attendue")
except ValueError:
    pass
assert [tuple(r) for r in helpdesk.execute("SELECT id, tech FROM ticket ORDER BY id")] == before, "rollback attendu"
audit = [r["action"] for r in helpdesk.execute("SELECT action FROM audit")]
assert audit == ["Youssef -> Hedi : 2 ticket(s)"]
assert helpdesk.execute("SELECT COUNT(*) FROM ticket WHERE tech = 'Hedi' AND status = 'OPEN'").fetchone()[0] == 3
print("✅ Exercice 9 réussi")

---
### Exercice de synthèse ⭐⭐⭐ – Import d'un CSV dans SQLite et export JSON

Le service achats fournit `achats.csv` (séparateur `;`, montants avec virgule décimale). Écrivez `import_purchases(csv_path, conn) -> tuple[int, list[str]]` :
- crée la table `purchase(id INTEGER PRIMARY KEY AUTOINCREMENT, supplier TEXT NOT NULL, item TEXT NOT NULL, quantity INTEGER NOT NULL CHECK(quantity > 0), unit_price REAL NOT NULL, ordered_on TEXT NOT NULL)` ;
- insère chaque ligne valide ; une ligne invalide (quantité ou prix non numérique, date non ISO, contrainte violée) est ignorée et signalée `"ligne N : <erreur>"` (N = numéro de ligne du fichier, en-tête = 1) ;
- toutes les insertions valides sont faites dans **une seule transaction**.

Puis `export_supplier_summary(conn, json_path)` écrit un JSON `{"generated": "2026-10-21", "suppliers": [{"supplier", "orders", "total"}...]}` trié par total décroissant (total arrondi à 3 décimales).

In [ ]:
from datetime import date


def import_purchases(csv_path: Path, conn: sqlite3.Connection) -> tuple[int, list[str]]:
    """Importe les achats valides ; renvoie (nombre importé, erreurs)."""
    # TODO


def export_supplier_summary(conn: sqlite3.Connection, json_path: Path, generated: date = date(2026, 10, 21)) -> dict:
    """Synthèse par fournisseur au format JSON."""
    # TODO


purchases_csv = WORK / "achats.csv"
purchases_csv.write_text("""fournisseur;article;quantite;prix_unitaire;date
TechnoPlus;Écran 27 pouces;4;649,000;2026-10-02
Bureautique Sud;Ramette A4;50;12,500;2026-10-05
TechnoPlus;Station d'accueil USB-C;6;389,900;2026-10-09
Bureautique Sud;Toner HP 59A;dix;215,000;2026-10-12
NetPro;Switch 48 ports;1;4100;2026-10-15
TechnoPlus;Clavier sans fil;10;45,9;15/10/2026
NetPro;Câble RJ45 Cat6 (lot);0;35,000;2026-10-18
""", encoding="utf-8")
purchases_db = sqlite3.connect(":memory:")
imported, errors = import_purchases(purchases_csv, purchases_db)
summary = export_supplier_summary(purchases_db, WORK / "fournisseurs.json")
print(imported, errors, json.dumps(summary, ensure_ascii=False, indent=1), sep="\n")

<details><summary>💡 Indice</summary>

Validez et convertissez chaque ligne dans un `try` (lecture du CSV), puis insérez toutes les lignes valides en une fois avec `executemany` dans un `with conn:`.

</details>

In [ ]:
# ✅ Vérification
assert imported == 4
assert [e.split(" : ")[0] for e in errors] == ["ligne 5", "ligne 7", "ligne 8"]
assert purchases_db.execute("SELECT COUNT(*) FROM purchase").fetchone()[0] == 4
saved = json.loads((WORK / "fournisseurs.json").read_text(encoding="utf-8"))
assert saved == {"generated": "2026-10-21", "suppliers": [
    {"supplier": "TechnoPlus", "orders": 2, "total": 4935.4},
    {"supplier": "NetPro", "orders": 1, "total": 4100.0},
    {"supplier": "Bureautique Sud", "orders": 1, "total": 625.0}]}
print("✅ Exercice de synthèse réussi")

---
## Projet fil rouge – étape 9 : CSV, JSON et persistance SQLite

Le modèle complet de l'étape 8 est fourni dans `scripts/gestparc/gestparc_modele.py` (chargé par `%run`).

**À faire :**
1. **CSV** : `clean_row(row)` convertit une ligne lue (tout en texte) selon `FIELD_TYPES` et **supprime les champs vides** ; `export_csv(inventory, path)` (séparateur `;`, colonnes `CSV_FIELDS`, `restval=""`) ; `import_csv(path)` → `(Inventory, erreurs)` avec des messages `"ligne N : ..."`.
2. **JSON** : `save_json(inventory, path)` → `{"app", "version", "equipments": [...]}` ; `load_json(path)` → `Inventory`.
3. **SQLite** : la classe `EquipmentRepository` (schéma fourni ; les attributs propres à chaque type sont stockés en JSON dans la colonne `specs`) :
   `save` (insertion **ou mise à jour** : `INSERT ... ON CONFLICT(asset_tag) DO UPDATE`), `save_all`, `get` (`EquipmentNotFoundError`), `delete`, `search(category, available_only)`, `count_by_category` (`GROUP BY`), `load_inventory`.

Toutes les requêtes sont **paramétrées**.

In [ ]:
%run scripts/gestparc/gestparc_modele.py
import csv
import json
import sqlite3

inventory = demo_inventory()
print(len(inventory), "équipements :", [e.asset_tag for e in inventory])

In [ ]:
CSV_FIELDS = ["asset_tag", "category", "brand", "model", "purchase_year", "purchase_date", "price",
              "assigned_to", "ram_gb", "screen_inches", "cpu_count", "rack", "is_color",
              "pages_printed", "port_count", "ip_address"]


FIELD_TYPES = {"purchase_year": int, "price": float, "ram_gb": int, "screen_inches": float,
               "cpu_count": int, "pages_printed": int, "port_count": int,
               "is_color": lambda value: value.strip().lower() in ("true", "1", "oui")}


def clean_row(row: dict[str, str]) -> dict:
    """Convertit une ligne CSV (tout en texte) : types, champs vides supprimés."""
    # TODO
    ...


def export_csv(inventory: Inventory, path: Path) -> int:
    """Exporte l'inventaire en CSV (séparateur ;) ; renvoie le nombre d'équipements écrits."""
    # TODO
    ...


def import_csv(path: Path) -> tuple[Inventory, list[str]]:
    """Importe un CSV ; renvoie l'inventaire et les erreurs (« ligne 3 : ... »)."""
    # TODO
    ...


def save_json(inventory: Inventory, path: Path) -> None:
    """Sauvegarde l'inventaire au format JSON (UTF-8, indenté)."""
    # TODO
    ...


def load_json(path: Path) -> Inventory:
    """Recharge un inventaire sauvegardé par save_json."""
    # TODO
    ...


class EquipmentRepository:
    """Persistance des équipements dans une base SQLite."""

    COMMON = ["asset_tag", "category", "brand", "model", "purchase_year", "purchase_date", "price", "assigned_to"]

    def __init__(self, db_path: str | Path = ":memory:") -> None:
        self.connection = sqlite3.connect(db_path)
        self.connection.row_factory = sqlite3.Row
        self.create_schema()

    def create_schema(self) -> None:
        with self.connection:
            self.connection.execute("""
                CREATE TABLE IF NOT EXISTS equipment (
                    asset_tag     TEXT PRIMARY KEY,
                    category      TEXT NOT NULL,
                    brand         TEXT NOT NULL,
                    model         TEXT NOT NULL,
                    purchase_year INTEGER NOT NULL,
                    purchase_date TEXT,
                    price         REAL NOT NULL CHECK (price >= 0),
                    assigned_to   TEXT,
                    specs         TEXT NOT NULL DEFAULT '{}'
                )""")

    def save(self, equipment: Equipment) -> None:
        """Insère ou met à jour un équipement (les attributs spécifiques sont stockés en JSON)."""
        # TODO
        ...

    def save_all(self, inventory: Inventory) -> int:
        # TODO
        ...

    @staticmethod
    def _to_equipment(row: sqlite3.Row) -> Equipment:
        data = {key: row[key] for key in EquipmentRepository.COMMON}
        data.update(json.loads(row["specs"]))
        return equipment_from_dict(data)

    def get(self, asset_tag: str) -> Equipment:
        # TODO
        ...

    def delete(self, asset_tag: str) -> None:
        # TODO
        ...

    def search(self, category: str | None = None, available_only: bool = False) -> list[Equipment]:
        # TODO
        ...

    def count_by_category(self) -> dict[str, int]:
        # TODO
        ...

    def load_inventory(self) -> Inventory:
        # TODO
        ...

    def close(self) -> None:
        self.connection.close()


print(export_csv(inventory, WORK / "inventaire.csv"))
print((WORK / "inventaire.csv").read_text(encoding="utf-8"))
reloaded, csv_errors = import_csv(WORK / "inventaire.csv")
save_json(reloaded, WORK / "inventaire.json")
repo = EquipmentRepository(WORK / "gestparc.db")
repo.save_all(load_json(WORK / "inventaire.json"))
print(repo.count_by_category(), [repr(e) for e in repo.search("laptop", available_only=True)])

In [ ]:
# ✅ Vérification
assert clean_row({"asset_tag": "LT-009", "price": "1200.5", "ram_gb": "16", "is_color": "", "rack": " R3 "}) == \
    {"asset_tag": "LT-009", "price": 1200.5, "ram_gb": 16, "rack": "R3"}
assert clean_row({"is_color": "True"})["is_color"] is True
lines = (WORK / "inventaire.csv").read_text(encoding="utf-8").splitlines()
assert len(lines) == 6 and lines[0].split(";") == CSV_FIELDS and lines[1].startswith("LT-001;laptop;Dell")
assert csv_errors == [] and len(reloaded) == 5
assert reloaded.get("SV-001").specs() == inventory.get("SV-001").specs()
assert reloaded.get("PR-001").ip_address == "10.0.2.50" and reloaded.get("LT-002").purchase_date == inventory.get("LT-002").purchase_date
bad_csv = WORK / "import_errone.csv"
bad_csv.write_text(lines[0] + "\nLT-010;laptop;Asus;B5;2024;;-5;;;;;;;;;\nXX-001;tablet;A;B;2024;;100;;;;;;;;;\nLT-011;laptop;HP;840;2024;;2100;;8;;;;;;;\n", encoding="utf-8")
inv_bad, errs = import_csv(bad_csv)
assert len(inv_bad) == 1 and [e.split(" : ")[0] for e in errs] == ["ligne 2", "ligne 3"]
data = json.loads((WORK / "inventaire.json").read_text(encoding="utf-8"))
assert data["app"] == "GestParc" and len(data["equipments"]) == 5
r = EquipmentRepository()                      # base en mémoire pour les tests
assert r.save_all(inventory) == 5 and r.count_by_category() == {"laptop": 2, "network": 1, "printer": 1, "server": 1}
lt = r.get("lt-002")
assert type(lt) is Laptop and lt.ram_gb == 16 and lt.is_available()
lt.assign("Sonia Haddad")
r.save(lt)
assert r.get("LT-002").assigned_to == "Sonia Haddad" and r.search("laptop", available_only=True) == []
assert r.get("SW-001").port_count == 48 and r.get("SV-001").ip_address == "10.0.0.10"
r.delete("PR-001")
assert "PR-001" not in r.load_inventory() and len(r.load_inventory()) == 4
for action in (lambda: r.get("PR-001"), lambda: r.delete("PR-001")):
    try:
        action()
        raise AssertionError("EquipmentNotFoundError attendue")
    except EquipmentNotFoundError:
        pass
repo.close()
print("✅ Étape 9 du projet GestParc validée")

✔️ Version de référence : `scripts/gestparc/main.py`, utilisable **en ligne de commande** :
```bash
export GESTPARC_DB=parc.db            # PowerShell : $env:GESTPARC_DB = "parc.db"
python main.py init                   # crée la base avec l'inventaire de démonstration
python main.py list laptop
python main.py export inventaire.json
python main.py import inventaire.csv
python main.py                        # menu interactif
```

**Prochaine étape (S14)** : ce fichier unique de plus de 600 lignes devient un **package** `gestparc` bien organisé.

---
## Pour aller plus loin (bonus)

1. **`argparse`** : réécrivez `parse_backup_args` avec `argparse.ArgumentParser` (aide `-h` générée automatiquement).
2. **Excel** : relisez `synthese_projets.csv` avec l'encodage `utf-8-sig` après l'avoir ouvert/enregistré dans Excel. Que change le BOM ?
3. **`subprocess`** : exécutez `python --version` depuis Python avec `subprocess.run([...], capture_output=True, text=True)`.
4. **SQL** : ajoutez une table `assignment_history` alimentée à chaque affectation dans `EquipmentRepository.save`.

In [ ]:
# À vous